<a href="https://www.kaggle.com/code/alinahurtova/practice-data-set-1?scriptVersionId=355881420" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import torch
from torchvision import datasets, transforms

In [ ]:
data_dir = "/kaggle/input/datasets/yapwh1208/cats-breed-dataset/cat_v1"

# Створіть екземпляр ImageFolder
dataset = datasets.ImageFolder(data_dir)

In [ ]:
len(dataset)

In [ ]:
img, label = dataset[100]

In [ ]:
img

In [ ]:
label

In [ ]:
dataset.classes

In [ ]:
from torchvision import transforms

In [ ]:
train_transformer = transforms.Compose([
    transforms.Resize([256,256]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
])

In [ ]:
train_transformer(img)

In [ ]:
train_transformer

In [ ]:
test_transformer = transforms.Compose([
    transforms.Resize([256,256]),
    transforms.ToTensor(),
])

In [ ]:
test_transformer

In [ ]:
from torch.utils.data import random_split

# Розділіть набір даних
train_dataset, test_dataset = random_split(dataset, [0.8, 0.2])

In [ ]:
len(train_dataset)

In [ ]:
len(test_dataset)

In [ ]:
# додати трансформери до дата сетів

from torch.utils.data import Dataset,DataLoader

class TransformDataset(Dataset):
    def __init__(self, dataset, transformer):
        super().__init__()
        self.dataset = dataset
        self.transformer = transformer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        img, label = self.dataset[index]

        img_new = self.transformer(img)

        return img_new, label

In [ ]:
train_dataset = TransformDataset(train_dataset, train_transformer)
test_dataset = TransformDataset(test_dataset, test_transformer)

In [ ]:
train_dataset[0]

In [ ]:
test_dataset[0]

In [ ]:
batch_size = 100

# Створіть завантажувачі даних - поділити на batch
train_loader = DataLoader(train_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
for images, label in train_loader:
    break

In [ ]:
images.shape

In [ ]:
from torch import nn

In [ ]:
dataset.classes

In [ ]:
len(dataset.classes)

In [ ]:
3*100*100

In [ ]:
# model = nn.Sequential(
#     nn.Flatten(),  # переведе зображення 3х100х100 в 30000 нейрони
#     nn.Linear(30000, 200),
#     nn.ReLU(),
#     nn.Linear(200, 50),
#     nn.ReLU(),
#     nn.Linear(50, 15),
#     nn.ReLU(),
#     nn.Linear(15, 5)   # 5 -- кількість порід собак(8 ймовірностей під кожну породу)
# )

model = nn.Sequential(
    nn.Conv2d(kernel_size=3, padding='same', out_channels=10, in_channels=3), #10, 256, 256
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2), #10, 128,128
    nn.Conv2d(kernel_size=3, padding='same', out_channels=20, in_channels=10), #20, 128, 128
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2), #20, 64,64
    nn.Conv2d(kernel_size=3, padding='same', out_channels=30, in_channels=20), #30, 64, 64
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2), #30, 32,32
    nn.Conv2d(kernel_size=3, padding='same', out_channels=30, in_channels=30), #30, 32, 32
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2), #30, 16,16

    nn.Flatten(),
    nn.Linear(7680,5) # 5 кількість порід котів
)

In [ ]:
30*16*16

In [93]:
for X_batch, y_batch in train_loader:
    break

In [ ]:
X_batch.shape

In [ ]:
X_batch[:1].shape

In [ ]:
model(X_batch)

In [ ]:
from torch.optim import Adam

# Функція втрат для класифікації
loss_fn = nn.CrossEntropyLoss()


# Оптимізатор (Adam) для оновлення ваг моделі
optimizer = Adam(
    model.parameters(),
    lr=0.001,
)

In [ ]:
# тренування

# підключення до відеокарти
device = "cuda"

model = model.to(device)

In [ ]:
# # epochs = 5  # епохи(скільки разів пройтись по всьому датасету під час тренування)

# # for _ in range(epochs):
# for X, y in train_loader:
#     # підключення до відеокарти
#     X = X.to(device)  # зображення
#     y = y.to(device)  # індекси породи
                                    
#     # отримати прогноз моделі
#     y_pred = model(X)
#     # обрахунок помилки - чим менше, тим точніше передбаченя моделі
#     loss = loss_fn(y_pred, y)
                                    
#     print(loss)
                                    
#     # покращення моделі
#     loss.backward()  # обрахунок градієнтів
#     optimizer.step()  # застосування градієнтів(зміна параметрів моделі)
#     optimizer.zero_grad()  #  градієнт очищається

In [ ]:
# epochs = 20  # епохи(скільки разів пройтись по всьому датасету під час тренування)

# list_of_train_losses = []
# list_of_test_losses = []

# for _ in range(epochs):
#     for X, y in train_loader:
#         # підключення до відеокарти
#         X = X.to(device)  # зображення
#         y = y.to(device)  # індекси породи
                                        
#         # отримати прогноз моделі
#         y_pred = model(X)
#         # обрахунок помилки - чим менше, тим точніше передбаченя моделі
#         loss = loss_fn(y_pred, y)
#         list_of_train_losses.append(loss)                                
#         print(loss.item())
        
                                        
#         # покращення моделі
#         loss.backward()  # обрахунок градієнтів
#         optimizer.step()  # застосування градієнтів(зміна параметрів моделі)
#         optimizer.zero_grad()  #  градієнт очищається
                                

In [ ]:
epochs = 20  # епохи(скільки разів пройтись по всьому датасету під час тренування)

train_losses = []
test_losses = []

for _ in range(epochs):

    # TRAIN
    for X_batch, y_batch in train_loader:
        # підключення до відеокарти
        X_batch = X_batch.to(device)  # зображення
        y_batch = y_batch.to(device)  # індекси породи
                                        
        # отримати прогноз моделі
        y_pred = model(X_batch)
        
        # обрахунок помилки - чим менше, тим точніше передбаченя моделі
        loss = loss_fn(y_pred, y_batch) 
        
        # зберігаємо loss КОЖНОГО train step
        train_losses.append(loss.item())
        print(loss.item())
        
                           
        # покращення моделі
        loss.backward()  # обрахунок градієнтів
        optimizer.step()  # застосування градієнтів(зміна параметрів моделі)
        optimizer.zero_grad()  #  градієнт очищається

    # TEST
    for X_batch, y_batch in test_loader: 
        # підключення до відеокарти
        X_batch = X_batch.to(device)  # зображення
        y_batch = y_batch.to(device)  # індекси породи
            
        # отримати прогноз моделі
        y_pred = model(X_batch)
            
        # обрахунок помилки - чим менше, тим точніше передбаченя моделі
        loss = loss_fn(y_pred, y_batch)
            
        test_losses.append(loss.item())
        print(loss.item())

In [ ]:
train_losses

In [ ]:
test_losses

In [ ]:
# графік
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(train_losses[:160], label="Train Loss")
plt.plot(test_losses[:160], label="Test Loss")

plt.xlabel("Steps")
plt.ylabel("Loss")

plt.xlim(0, 160)
plt.ylim(0, 2)

plt.legend()
plt.grid()

plt.show()